# FairnessBench Results

All performance + flake8 results for the paper, in one place. Each section maps to a
figure/table (see `readme.md` for the paper-section mapping). Shared logic lives in the
importable modules:

- `loading.py` — CSV loaders (`load_performance`, `load_baseline`, `load_dollarstreet`,`load_run_counts`,`load_perf_and_baseline`)
- `cleaning.py` — task decomposition, target-metric extraction, baseline improvement,
  interval-overlap sensitivity, Pareto trade-off math
- `plotting.py` — reference lines, grouped heatmaps, figure saving
- `constants.py` — metric maps/ideals, model renames, dataset/prompt variant vocabulary

**Data**: every section below works off the single main `Final_step_perfomance*.csv`
loaded once in section 1 (plus the baseline CSV where a comparison against unmodified
`train.py` is needed). Except for Dollarstreet anaylsis works off `Dollarstreet_performance*.csv`, and runs count analysis works on the `run_counts*.csv` The loaders pick the newest file in `csv_files/` automatically,
so after `python explode_results.py` there is nothing to edit here; pin an older file
with `constants.PERFORMANCE_CSV` to reproduce a published figure exactly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cleaning
from constants import (
    METRIC_MAP, METRIC_DISPLAY, METRIC_IDEAL, PERF_COLS, FAIRNESS_COLS,
    HIGH_GOOD, LOW_GOOD, VARIANT_BASELINE, VARIANT_DATASETS,
    ALLOWED_PROMPT_VARIATIONS, INCOME_COLS,
)
from loading import (load_performance, load_baseline, load_dollarstreet, load_run_counts,
                     load_perf_and_baseline)
from plotting import (
    save_fig, ref_lines, metric_ref_lines, rotate_xticklabels,
    grouped_heatmap, symmetric_limits, change_heatmap,
)

## 1. Load the results

One row per (model, task, run): accuracy, the six fairness metrics, and the flake8
fairness-plugin score. Task IDs are already decomposed into `dataset`, `task_metric`,
`research_problem`, and `dem` (the sensitive attribute). Every later section reuses
this frame.

### Run counts per model × research problem

## 2. Fairness vs. accuracy across datasets — Figures 3 & 13 (S4.2.1)

Scatter of the targeted fairness metric against accuracy, per model and research
problem. Reference lines mark the metric's ideal value and accuracy = 1.

In [ ]:
di_runs = perf[perf['task_metric'] == 'di']
di_fail = (
    di_runs.groupby(['model', 'dataset'])['di']
    .agg(fail=lambda x: (x < .8).sum(), count='size', rate=lambda x: (x < .8).mean())
)
di_fail['rate'].unstack().T.to_latex('di_fail.tex')
di_fail['rate'].unstack().T

In [ ]:
sns.set_context(context='poster', font_scale=1.5)
for metric in ['di', 'spd', 'eod']:
    sub = perf[perf['task_metric'] == metric]
    sub=sub.rename(columns={'research_problem':'RP'})
    g = sns.relplot(data=sub, x='acc', y=METRIC_MAP[metric], hue='dataset',
                    row='RP', style='dem', col='model',
                    kind='scatter', aspect=1.2)
    g.set_titles(template='{col_var}: {col_name}\n{row_var}: {row_name}')
    ref_lines(g, h=METRIC_IDEAL[metric], v=1.0)
    save_fig(f'{metric}_vs_acc_scatter.png')

## 3. Adult dataset: targeted fairness metrics — Figure 4 (S 4.2.2)

Mean value of each targeted metric on the Adult tasks, by research problem and
sensitive attribute. Dashed lines mark each metric's ideal value.

In [ ]:
adult = perf[perf['dataset'] == 'adult']
adult_mean = (
    adult.groupby(['model', 'research_problem', 'dem', 'task_metric'])[FAIRNESS_COLS]
    .mean().reset_index()
)
adult_mean = cleaning.add_task_metric_value(adult_mean)
adult_mean=adult_mean.rename(columns={'research_problem':'RP'})
sns.set_context(context='poster', font_scale=1.3)

g = sns.catplot(data=adult_mean, x='RP', y='task_metric_value',
                hue='dem', row='task_metric', col='model', kind='bar', aspect=1)
g.set_titles(template='{col_var}: {col_name}\n{row_var}: {row_name}')
metric_ref_lines(g, g.row_names, v=None)
rotate_xticklabels(g)
save_fig('adult_fairness.png')

## 4. Balancing fairness and accuracy — Figure 6 (S4.3)

Per-run targeted-metric value vs. accuracy for the `balance` research problem
(main figure: DI and EOD), plus the full metric grids for `best` and `implicit`.

In [ ]:
perf_tm = cleaning.add_task_metric_value(perf)

main_df = perf_tm[(perf_tm['research_problem'] == 'balance')
                  & perf_tm['task_metric'].isin(['di', 'eod'])]

sns.set_context(context='poster', font_scale=1.5)
g = sns.relplot(data=main_df, x='acc', y='task_metric_value', hue='dataset',
                style='dem', row='task_metric', col='model', kind='scatter',
                aspect=1, s=120, facet_kws={'sharey': 'row'})
g.set_titles(template='{col_name}\n{row_name}')
g.set_axis_labels('Accuracy', '')
for ax_row, metric in zip(g.axes, g.row_names):
    ax_row[0].set_ylabel(METRIC_DISPLAY[metric])
metric_ref_lines(g, g.row_names)
save_fig('balancing_fairness.png', dpi=400)

In [ ]:
sns.set_context(context='poster', font_scale=1.3)
for rp in ['best', 'implicit']:
    sub = perf_tm[perf_tm['research_problem'] == rp]
    g = sns.relplot(data=sub, x='acc', y='task_metric_value', hue='dataset',
                    style='dem', row='task_metric', col='model', kind='scatter', aspect=1)
    g.set_titles(template='{col_var}: {col_name}\n{row_var}: {row_name}')
    metric_ref_lines(g, g.row_names)
    save_fig(f'balancing_fairness_{rp}.png', dpi=400)

## 5. Target10: improvement over baseline — Figure 5 (S4.2.3)

For the `target10` research problem (improve the metric by 10%), each run is scored
against the unmodified `train.py` baseline. `success` = improved the targeted metric by
more than 0.1; `improvement` = any improvement at all.

In [ ]:
merged = load_perf_and_baseline()
t10 = cleaning.add_improvement(merged[merged['research_problem'] == 'target10'].copy())

t10_stats = (
    t10.groupby(['model', 'dataset'])['agent-improvement']
    .agg(mean='mean',
         success=lambda s: (s > .1).sum(),
         total='size',
         improvement=lambda s: (s > 0).sum())
    .reset_index()
)
display(t10_stats)

t10_tall = t10_stats.melt(id_vars=['model', 'dataset'],
                          value_vars=['total', 'success', 'improvement'],
                          var_name='count_type', value_name='count')
sns.set_context('paper', font_scale=1.8)
g = sns.catplot(data=t10_tall, x='dataset', y='count', hue='count_type',
                col='model', kind='bar', height=5, aspect=1.0, legend_out=True)
rotate_xticklabels(g, rotation=45, ha='right')
save_fig('target10_success.png', dpi=400)


## 6. Accuracy–fairness trade-off heatmap — Figure 7 (S4.3)

For each (dataset, model, research problem) we take the Pareto front over accuracy and
a DI-based fairness score, and summarize the front's mean angle relative to the 45°
line: negative (purple) leans toward accuracy, positive (green) toward fairness. Cells
whose front has < 2 points have no meaningful trade-off and are blank.

In [ ]:
tradeoff = perf[np.isfinite(perf['di'])
                & perf['research_problem'].isin(['balance', 'implicit', 'best'])].copy()
tradeoff['fair'] = (1 - (tradeoff['di'] - 1).abs()).clip(lower=0)

summary, pareto_df = cleaning.pareto_tradeoff_summary(tradeoff)
pivot = summary.pivot(index=['research_problem', 'model'], columns='dataset',
                      values='theta_centered')
vmin, vmax = symmetric_limits(pivot.values)
grouped_heatmap(pivot, xlabel='Dataset',
                cbar_label='Centered Angle from 45° (degrees)',
                cmap='PRGn_r', center=0, vmin=vmin, vmax=vmax, fmt='.1f',
                figsize=(11, 8), annot_size=11, group_label_x=-0.18)
save_fig('acc_di_tradeoff_heatmap.png')

## 7. Sensitivity to dataset variants — Figures 8 & 12 (S4.4)

Each variant dataset (`randoadult`, `sampadult`, `nondescriptive`, `health`, `samp*`)
is compared to its source dataset. For accuracy and DI separately, a variant is
**similar** when its mean falls inside the source dataset's mean ± std band, and
otherwise **better** or **worse** (DI by distance from its ideal of 1). Each cell's
colour is the accuracy/fairness pair; symbols flag degenerate comparisons.

In [ ]:
ds_summary = cleaning.mean_std_summary(perf, ['model', 'dataset', 'research_problem'])
ds_sens = cleaning.overlap_vs_baseline(ds_summary, item_col='dataset',
                                       baseline=VARIANT_BASELINE,
                                       allowed=VARIANT_DATASETS)
ds_sens = cleaning.add_change_labels(ds_sens)

change_heatmap(ds_sens, 'comparison_dataset', xlabel='Comparison Dataset',
               title='Accuracy and Fairness (DI) Change vs. Baseline Dataset')
save_fig('fair_acc_sensitivity_di.png', dpi=400)

## 8. Sensitivity to prompt variations — Figures 10 & 11 (S4.6)

Same better / similar / worse comparison, with each rephrased or altered prompt for
`adult_balance-eod-sex` compared against the original prompt.

In [ ]:
prompt_df = cleaning.prompt_sensitivity_frame(perf)
prompt_df = prompt_df[prompt_df['prompt_variation'] != 'noreq']
count = prompt_df.groupby(['model', 'prompt_variation'])["research_problem"].count()
display(count.reset_index())
 
pv_summary = cleaning.mean_std_summary(
    prompt_df, ['model', 'research_problem', 'prompt_variation'])
pv_sens = cleaning.overlap_vs_baseline(pv_summary, item_col='prompt_variation',
                                       baseline='original',
                                       allowed=ALLOWED_PROMPT_VARIATIONS)
pv_sens = cleaning.add_change_labels(pv_sens)

change_heatmap(pv_sens, 'comparison_prompt_variation', xlabel='Comparison Prompt',
               title='Accuracy and Fairness (DI) Change vs. Baseline Prompt')
save_fig('prompt_sensitivity_di.png', dpi=400)

## 9. Target selection (adrecon) — Figure 9 (S4.5.1)

On the `targetselection` tasks the agent chooses which metric to optimise, so every
metric is reported. Each point is one run: its accuracy against one fairness metric,
by model and sensitive attribute. The dashed line marks each metric's ideal value
(1 for the ratios, 0 for the differences).

In [ ]:
adrecon = perf[perf['dataset'] == 'adrecon'].reset_index(drop=True)
print('% of runs with all metrics reported, per model:')
display((adrecon[PERF_COLS].notna().all(axis=1).groupby(adrecon['model']).mean() * 100).round(2))
adrecon = adrecon.dropna(subset=PERF_COLS)

short = {'statistical_parity_diff': 'spd', 'equal_opp_diff': 'eod',
         'error_rate_diff': 'erd', 'error_rate_ratio': 'err',
         'false_omission_rate_diff': 'ford'}
adrecon = adrecon.rename(columns=short)
metric_cols = ['di', 'eod', 'spd', 'err', 'erd', 'ford']

plot_df = adrecon.melt(id_vars=['model', 'dem', 'acc'], value_vars=metric_cols,
                       var_name='task_metric', value_name='task_metric_value')

sns.set_context('paper')
g = sns.relplot(data=plot_df, x='acc', y='task_metric_value', hue='dem',
                row='task_metric', row_order=metric_cols, col='model',
                kind='scatter', aspect=1.1, height=3, s=55,
                facet_kws={'sharey': False})
g.set_titles(row_template='{row_name}', col_template='{col_name}', size=10)
g.set_axis_labels('Accuracy', '')
for ax in g.axes.flat:
    ax.tick_params(axis='both', labelsize=8)
for ax_row, metric in zip(g.axes, metric_cols):
    ax_row[0].set_ylabel(METRIC_DISPLAY[metric], fontsize=9)

metric_ref_lines(g, metric_cols, v=None)
g.fig.subplots_adjust(top=0.93, hspace=0.45, wspace=0.25)
save_fig('adrecon_allmetric.png', dpi=400)

## 10. DollarStreet: accuracy by income level ( S4.5.2)

Accuracy on images from advantaged vs. disadvantaged income groups, for the agents and
the unmodified `train.py` baseline. Table only (no paper figure).

In [ ]:
dollar = load_dollarstreet()
print(dollar.attrs['source'])

base = load_baseline()
base_income = ['baseline_' + c for c in INCOME_COLS]
dollar_base = (base[base[base_income].notna().all(axis=1)][['task'] + base_income]
               .rename(columns=dict(zip(base_income, INCOME_COLS)))
               .assign(model='baseline'))

dollar_all = pd.concat([dollar_base, dollar[['model', 'task'] + INCOME_COLS]], ignore_index=True)
dollar_avg = (
    dollar_all.groupby('model')
    .agg(avg_adv_acc=('Advantaged', 'mean'), avg_disadv_acc=('Disadvantaged', 'mean'),
         std_adv_acc=('Advantaged', 'std'), std_disadv_acc=('Disadvantaged', 'std'),
         n_runs=('Advantaged', 'count'))
    .reset_index()
)
dollar_avg['disparity'] = dollar_avg['avg_adv_acc'] - dollar_avg['avg_disadv_acc']
display(dollar_avg)

## 11. Flake8 fairness-plugin scores across research problems ( S4.3)

Mean flake8 plugin score (max 100, dashed line) for `balance`, `best`, and `implicit`,
per model and dataset.

In [ ]:
f8 = perf[perf['research_problem'].isin(['balance', 'implicit', 'best'])].copy()
f8['final_flake8_score'] = pd.to_numeric(f8['final_flake8_score'], errors='coerce')
f8=f8.rename(columns={'research_problem':'RP'})
sns.set_context(context='poster', font_scale=0.8)
g = sns.catplot(data=f8, x='RP', y='final_flake8_score',
                col='model', row='dataset', kind='bar', height=4, aspect=1)
g.set_titles(template='{col_var}: {col_name}\n{row_var}: {row_name}')
ref_lines(g, h=100, alpha=1.0)
save_fig('comparing_flake8_bal_be_impli.png')

## 12. Run and success counts — Tables 10 (appendix)

How often each model produced a usable result. These counts come from
`Run_counts*.csv`, which is built from every run including the failures, so the
denominators are complete — the performance CSV only keeps runs that were scored.

| column | meaning |
|---|---|
| `runs` | attempted |
| `completed_runs` | finished without an error (`total_time > 0`, empty `error`) |
| `successful_runs` | produced a `submission.csv` that could be scored |
| `failed_runs` | `runs - successful_runs`: no scoreable submission, for any reason |
| `completion_rate` | `completed_runs / runs` — got to the end of the agent loop |
| `submission_rate` | `successful_runs / runs` — the quantity Table 10 reports |

A run can complete and still fail to submit (it finished, but wrote no usable
`submission.csv`), so `completion_rate` is always at least `submission_rate`.

The `failed_*` columns say *why* the failures failed. `eval.py` tags each run by
string-matching its logs, and a tag can be set on a run that still succeeded, so these
count only runs that produced no score. A run can carry more than one tag, so the
columns overlap and do not sum to `failed_runs`; `failed_unexplained` is the failures
whose logs carry no tag at all.

In [ ]:
counts = load_run_counts()
print(counts.attrs['source'])

# Table 4: totals per model and research problem
by_rp = (counts.groupby(['model', 'research_problem'])[['runs', 'completed_runs', 'successful_runs']]
         .sum().reset_index())
by_rp['failed_runs'] = by_rp['runs'] - by_rp['successful_runs']
by_rp['completion_rate'] = by_rp['completed_runs'] / by_rp['runs']
by_rp['submission_rate'] = by_rp['successful_runs'] / by_rp['runs']
by_rp['failure_rate'] = by_rp['failed_runs'] / by_rp['runs']
by_rp = by_rp[['model', 'research_problem', 'runs', 'completed_runs', 'successful_runs',
               'failed_runs', 'completion_rate', 'submission_rate', 'failure_rate']]
display(by_rp)

# Table 10: rates, models x research problems
for value in ['submission_rate', 'completion_rate']:
    table = by_rp.pivot(index='model', columns='research_problem', values=value)
    print(f'\n{value}:')
    display(table.round(3))
    print(table.to_latex(float_format='%.3f'))

print(by_rp.to_latex(index=False, float_format='%.3f'))

# Why the failures failed
failure_cols = [c for c in counts.columns if c.startswith('failed_')]
failures = counts.groupby('model')[['runs', 'successful_runs'] + failure_cols].sum()
failures.insert(0, 'failed_runs', failures['runs'] - failures['successful_runs'])
failures = failures.drop(columns=['runs', 'successful_runs'])
display(failures)
print(failures.to_latex())